In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import pickle
import torch

import SASRec_class as sasrec

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

## 1 Load Processed Data

In [2]:
datasets = ["ml-1m", 'steam', "goodreads", "ml-10m"]
DATASET = datasets[3]

In [ ]:
base_artifacts = Path.cwd().parents[1] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Processed' / DATASET
data = pd.read_csv(
    data_path / 'data_clean.csv'
)
print(f"Data shape: {data.shape}")

with open(data_path / 'item_dict.pkl', 'rb') as f:
    item_dict = pickle.load(f)

with open(data_path / 'test_users.pkl', 'rb') as f:
    test_users = pickle.load(f)

unique_users = data['user_id'].unique()
train_users = [user for user in unique_users if user not in test_users]

Data shape: (3392139, 3)


In [4]:
# History length
L = 50

# 2 Prepare Windows for Training

In [5]:
users_dict = data.groupby('user_id')['item_id'].apply(list).to_dict()

In [6]:
padding_idx = data['item_id'].max() + 1

train_dataset = []
test_dataset = []
for user_id in tqdm(unique_users):
    padded_sequence = [padding_idx] * (L - 2) + users_dict[user_id]
    for i in range(len(padded_sequence) - L + 1):
        window = padded_sequence[i:i+L]
        if user_id in train_users:
            train_dataset.append(window)
        else:
            test_dataset.append(window)

train_dataset = np.array(train_dataset)
test_dataset = np.array(test_dataset)

  0%|          | 0/5369 [00:00<?, ?it/s]

In [7]:
if DATASET == "ml-10m":
    rng = np.random.default_rng(seed=42)

    train_random_idx = rng.choice(len(train_dataset), size=1_000_000, replace=False)
    train_dataset = train_dataset[train_random_idx]

    test_random_idx = rng.choice(len(test_dataset), size=200_000, replace=False)
    test_dataset = test_dataset[test_random_idx]

# 3 Train the model

In [8]:
model = sasrec.SASRecTorch(
    num_items=padding_idx+1,
    max_seq_len=L,
    d_model=50,
    n_heads=1,
    n_layers=2,
    dropout=0.5,
    device="cuda",
)
model.fit(
    train_dataset=train_dataset,
    valid_dataset=test_dataset,
    batch_size=2**11,
    lr=1e-3,
    weight_decay=0.0, 
    num_epochs=40,
)

/home/gouni/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch | T-Loss | V-Loss | Pctl  | HR10  | NDCG  | Cosθ  | Elapsed Time
======|========|========|=======|=======|=======|=======|=============
    1 |  1.213 |  1.050 | 0.784 | 0.397 | 0.324 | None  |     00:47.7
    2 |  0.996 |  0.900 | 0.849 | 0.540 | 0.388 | 0.472 |     01:35.1
    3 |  0.917 |  0.845 | 0.869 | 0.597 | 0.419 | 0.491 |     02:22.4
    4 |  0.868 |  0.795 | 0.883 | 0.644 | 0.447 | 0.548 |     03:09.7
    5 |  0.840 |  0.775 | 0.888 | 0.662 | 0.458 | 0.616 |     03:57.0
    6 |  0.823 |  0.755 | 0.893 | 0.675 | 0.467 | 0.645 |     04:44.3
    7 |  0.809 |  0.745 | 0.896 | 0.687 | 0.476 | 0.650 |     05:31.6
    8 |  0.799 |  0.737 | 0.898 | 0.694 | 0.481 | 0.630 |     06:19.0
    9 |  0.792 |  0.731 | 0.900 | 0.698 | 0.485 | 0.586 |     07:06.6
   10 |  0.787 |  0.725 | 0.901 | 0.702 | 0.488 | 0.528 |     07:54.0
   11 |  0.782 |  0.722 | 0.902 | 0.706 | 0.491 | 0.469 |     08:41.4
   12 |  0.779 |  0.719 | 0.903 | 0.708 | 0.492 | 0.414 |     09:28.7
   13 |  0.776 |  

In [9]:
folder_path = base_artifacts / 'SASRec_Models' / f'{DATASET}'
model.save(path=folder_path / f'sasrec.pt')

init_dict = {
    "num_items": padding_idx+1,
    "max_seq_len": L,
    "d_model": model.d_model,
    "n_heads": model.n_heads,
    "n_layers": model.n_layers,
    "dropout": model.dropout,
    "device": model.device
}

with open(folder_path / f'init_dict.pkl', 'wb') as f:
    pickle.dump(init_dict, f)

# 4 Load a Model

In [10]:
folder_path = base_artifacts / 'SASRec_Models' / f'{DATASET}'
with open(folder_path / f'init_dict.pkl', 'rb') as f:
    init_dict_loaded = pickle.load(f)

loaded_model = sasrec.SASRecTorch(**init_dict_loaded)
loaded_model.load(folder_path / f'sasrec.pt')

Model loaded from /home/gouni/CausalI2I_new_artifacts/SASRec_Models/ml-10m/sasrec.pt.
num_items:     3363
max_seq_len:   50
device:        cuda
batch_size:    2048
lr:            0.001
weight_decay:  0.0
num_epochs:    40
saved_at:      2026-07-24 16:40:57
note:          None


/home/gouni/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
